# ChaoSSM — Phase 23: robustness of the frozen-reservoir ordering

Addresses **Reviewer 2 comment 4**: is the central frozen-reservoir result an artifact of the one
architecture it was measured on (width N=64, LayerNorm features, a linear ridge readout, a single
reservoir layer), or does it survive changes to those choices?

**The result under test.** Under the canonical reservoir-computing protocol (recurrence *and* input
weights frozen and random, only the readout fit), three orderings should hold:
- **nonlinearity helps**: the edge reservoir beats the matched linear reservoir (`edge − linear > 0`);
- **criticality is not privileged**: the ordered reservoir matches the edge one (`ordered − edge ≈ 0`);
- **deep chaos collapses**: the edge reservoir beats the deep-chaos one (`edge − deep > 0`).

**What this notebook does.** Starting from the paper's baseline (N=64, LayerNorm, ridge readout,
depth 1), it varies **one factor at a time** and re-measures all four arms (ordered / edge / deep /
linear) at 5 seeds:
- **width** N ∈ {32, 64, 128} (input dimension scales with N),
- **normalization** ∈ {LayerNorm, none, per-feature z-score},
- **readout** ∈ {linear ridge, 1-hidden-layer MLP head},
- **depth** ∈ {1, 2 stacked frozen reservoirs}.

The ridge readout keeps every cell deterministic and instant to fit; Phase 19 established that ridge
and gradient-descent readouts give the same arm ordering, so ridge alone is a sound robustness probe.
Calibrated γ values are cached per (arm, N, seed), so each reservoir is Lyapunov-calibrated once.

Runtime: ~25–45 min on a T4 (N=128 calibration dominates). Resumable.

## Setup

In [1]:
import os, json, math, time, warnings, itertools
from dataclasses import dataclass, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))

def setup_results_dir(project="ChaoSSM",phase="phase23_robustness"):
    base=None
    try:
        from google.colab import drive
        try: drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}); retrying force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78); print(f"[WARNING] Google Drive NOT mounted ({type(e).__name__}). Results -> {base}")
        print("[WARNING] Local disk, lost on recycle. Run the rescue cell or download."); print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=600,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)

torch 2.11.0+cu130 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase23_robustness


## Configuration

In [2]:
SMOKE_TEST=False

@dataclass
class Config:
    r_cap_map:float=4.0; gamma_init:float=1.0
    calib_steps:int=200; calib_warm:int=60; lambda_tol:float=0.05
    lambda_target:dict=field(default_factory=lambda: dict(ordered=-0.30, edge=-0.02, deep=0.30))
    arms:tuple=("ordered","edge","deep","linear")
    seeds:int=5
    fc_length:int=64; fc_bins:int=32
    ridge_streams:int=150; ridge_alpha:float=1.0; eval_n:int=800; eval_seed:int=98765
    mlp_hidden:int=128; mlp_steps:int=400; mlp_lr:float=3e-3
    # baseline architecture
    base_N:int=64; base_norm:str="layernorm"; base_readout:str="ridge"; base_depth:int=1
    # sweep axes (one factor at a time off the baseline)
    sweep_N:tuple=(32,64,128)
    sweep_norm:tuple=("layernorm","none","zscore")
    sweep_readout:tuple=("ridge","mlp")
    sweep_depth:tuple=(1,2)
CFG=Config()
if SMOKE_TEST:
    CFG.seeds=2; CFG.calib_steps=30; CFG.calib_warm=10; CFG.ridge_streams=30; CFG.eval_n=80
    CFG.mlp_steps=40; CFG.sweep_N=(32,64); CFG.sweep_norm=("layernorm","none"); CFG.arms=("edge","deep","linear")
    print("*** SMOKE TEST — results meaningless ***")
print("arms:",CFG.arms,"| seeds:",CFG.seeds)

arms: ('ordered', 'edge', 'deep', 'linear') | seeds: 5


## Cells, calibration (verbatim from Phase 14/19; width made a parameter)

In [3]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,N,d_model,cfg):
        super().__init__(); self.N=N; self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(N))
        W=torch.empty(N,N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(N,d_model)/math.sqrt(d_model))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze(self):
        for p_ in self.parameters(): p_.requires_grad_(False)

class LinearCell(nn.Module):
    def __init__(self,N,d_model,cfg):
        super().__init__(); self.N=N
        W=torch.empty(N,N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(N,d_model)/math.sqrt(d_model))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze(self):
        for p_ in self.parameters(): p_.requires_grad_(False)

from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0):
    steps,warm,N=cfg.calib_steps,cfg.calib_warm,cell.N; dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0,1)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    target=cfg.lambda_target[regime]; lo,hi=0.01,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1]

GAMMA_CACHE={}      # (arm,N,seed) -> (gamma, lam)
def build_layer(cfg,arm,N,d_model,seed,layer_idx):
    """Deterministic frozen cell for one layer. Chaotic arms calibrated (gamma cached)."""
    torch.manual_seed(seed*100+layer_idx); np.random.seed(seed*100+layer_idx)
    if arm=="linear":
        cell=LinearCell(N,d_model,cfg).to(DEVICE); lam=float("nan")
    else:
        cell=ChaosCell(N,d_model,cfg).to(DEVICE)
        key=(arm,N,seed,layer_idx)
        if key in GAMMA_CACHE:
            gval,lam=GAMMA_CACHE[key]; cell.set_gamma(gval)
        else:
            gval,lam=calibrate_gamma(cell,cfg,arm,seed=seed*100+layer_idx); GAMMA_CACHE[key]=(gval,lam)
    cell.freeze(); return cell,lam

## Task, multi-layer rollout, normalizations, readouts

In [4]:
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt

@torch.no_grad()
def collect_states(cells,encoder,x):
    """Roll x through a stack of frozen cells; return the TOP layer's state sequence (B,T,N)."""
    e=encoder(x.unsqueeze(-1)); B,T=e.shape[0],e.shape[1]
    seq=e
    for cell in cells:
        h=cell.init_state(B,seq.device,seq.dtype); H=[]
        for t in range(T): h=cell.step(h,seq[:,t]); H.append(h)
        seq=torch.stack(H,1)
    return seq

def normalize(feat,kind,stats=None):
    """feat: (M,N). layernorm = per-row over features; zscore = per-column using train stats."""
    if kind=="none": return feat,None
    if kind=="layernorm": return F.layer_norm(feat,(feat.shape[-1],)),None
    if kind=="zscore":
        if stats is None: mu=feat.mean(0,keepdim=True); sd=feat.std(0,keepdim=True)+1e-6; stats=(mu,sd)
        mu,sd=stats; return (feat-mu)/sd,stats
    raise ValueError(kind)

@torch.no_grad()
def readout_ridge(cfg,Ftr,Ytr,Fev,yev):
    X=torch.cat([Ftr,torch.ones(Ftr.shape[0],1,device=Ftr.device)],1)
    A=X.T@X+cfg.ridge_alpha*torch.eye(X.shape[1],device=X.device)
    Wr=torch.linalg.solve(A,X.T@Ytr)
    EX=torch.cat([Fev,torch.ones(Fev.shape[0],1,device=Fev.device)],1)
    return float((( EX@Wr).argmax(-1)==yev).float().mean())

def readout_mlp(cfg,Ftr,ytr,Fev,yev,seed):
    torch.manual_seed(seed+7)
    net=nn.Sequential(nn.Linear(Ftr.shape[1],cfg.mlp_hidden),nn.ReLU(),
                      nn.Linear(cfg.mlp_hidden,cfg.fc_bins)).to(Ftr.device)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.mlp_lr)
    for _ in range(cfg.mlp_steps):
        opt.zero_grad(set_to_none=True)
        L=F.cross_entropy(net(Ftr),ytr); L.backward(); opt.step()
    with torch.no_grad(): acc=float((net(Fev).argmax(-1)==yev).float().mean())
    return acc

## One run = one (architecture variant, arm, seed) -> held-out accuracy

In [5]:
def run_config(cfg,N,norm,readout,depth,arm,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    d_model=N
    encoder=nn.Linear(1,d_model).to(DEVICE)
    for p_ in encoder.parameters(): p_.requires_grad_(False)
    cells=[]; lam_top=float("nan")
    for li in range(depth):
        in_dim=d_model if li==0 else N
        cell,lam=build_layer(cfg,arm,N,in_dim,seed,li); cells.append(cell); lam_top=lam
    # features
    gtr=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gtr.manual_seed(seed)
    itr,ttr=make_forecast(cfg,cfg.ridge_streams,DEVICE,gtr)
    gev=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gev.manual_seed(cfg.eval_seed)
    iev,tev=make_forecast(cfg,cfg.eval_n,DEVICE,gev)
    Ftr=collect_states(cells,encoder,itr).reshape(-1,N)
    Fev=collect_states(cells,encoder,iev).reshape(-1,N)
    ytr=ttr.reshape(-1); yev=tev.reshape(-1)
    Ftr,stats=normalize(Ftr,norm); Fev,_=normalize(Fev,norm,stats)
    if readout=="ridge":
        Ytr=F.one_hot(ytr,cfg.fc_bins).float(); acc=readout_ridge(cfg,Ftr,Ytr,Fev,yev)
    else:
        acc=readout_mlp(cfg,Ftr,ytr,Fev,yev,seed)
    return acc,lam_top

## Run the one-factor-at-a-time sweep

In [6]:
CKPT=os.path.join(RESULTS,"phase23_robustness.csv")
COLS=["variant","axis","N","norm","readout","depth","arm","seed","lambda","acc"]
if os.path.exists(CKPT):
    rows=pd.read_csv(CKPT).to_dict("records"); done={(r["variant"],r["arm"],int(r["seed"])) for r in rows}
    print(f"[resume] {len(rows)} runs done")
else: rows=[]; done=set()
def _flush(): pd.DataFrame(rows)[COLS].to_csv(CKPT,index=False)

# build the list of architecture variants: baseline + one-factor-at-a-time
B=dict(N=CFG.base_N,norm=CFG.base_norm,readout=CFG.base_readout,depth=CFG.base_depth)
variants=[]
def add(axis,**over):
    v=dict(B); v.update(over)
    name=f"{axis}={over[list(over)[0]]}" if over else "baseline"
    variants.append((name,axis,v))
add("baseline")
for N in CFG.sweep_N:
    if N!=CFG.base_N: add("width",N=N)
for nm in CFG.sweep_norm:
    if nm!=CFG.base_norm: add("norm",norm=nm)
for ro in CFG.sweep_readout:
    if ro!=CFG.base_readout: add("readout",readout=ro)
for dp in CFG.sweep_depth:
    if dp!=CFG.base_depth: add("depth",depth=dp)
print(f"{len(variants)} architecture variants × {len(CFG.arms)} arms × {CFG.seeds} seeds\n")

t0=time.time()
for name,axis,v in variants:
    for arm in CFG.arms:
        for sd in range(CFG.seeds):
            if (name,arm,sd) in done: continue
            acc,lam=run_config(CFG,v["N"],v["norm"],v["readout"],v["depth"],arm,sd)
            rows.append(dict(variant=name,axis=axis,N=v["N"],norm=v["norm"],readout=v["readout"],
                             depth=v["depth"],arm=arm,seed=sd,**{"lambda":lam},acc=acc)); _flush()
    sub=pd.DataFrame([r for r in rows if r["variant"]==name])
    mu=sub.groupby("arm").acc.mean()
    print(f"[P23] {name:14s} "+" ".join(f"{a}={mu.get(a,float('nan')):.3f}" for a in CFG.arms))
RES=pd.DataFrame(rows)[COLS]; _flush()
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")

7 architecture variants × 4 arms × 5 seeds

[P23] baseline       ordered=0.355 edge=0.330 deep=0.218 linear=0.250
[P23] width=32       ordered=0.297 edge=0.280 deep=0.202 linear=0.240
[P23] width=128      ordered=0.487 edge=0.422 deep=0.234 linear=0.254
[P23] norm=none      ordered=0.311 edge=0.299 deep=0.218 linear=0.249
[P23] norm=zscore    ordered=0.374 edge=0.340 deep=0.218 linear=0.253
[P23] readout=mlp    ordered=0.749 edge=0.706 deep=0.303 linear=0.516
[P23] depth=2        ordered=0.405 edge=0.338 deep=0.180 linear=0.248

wall clock: 3.9 min


## Analysis — is the ordering preserved under every variation?

In [7]:
def gaps(sub):
    mu=sub.groupby("arm").acc.mean()
    g={}
    if "edge" in mu and "linear" in mu: g["edge-linear"]=mu["edge"]-mu["linear"]
    if "ordered" in mu and "edge" in mu: g["ordered-edge"]=mu["ordered"]-mu["edge"]
    if "edge" in mu and "deep" in mu: g["edge-deep"]=mu["edge"]-mu["deep"]
    return mu,g

summary=[]
for name,axis,v in variants:
    sub=RES[RES.variant==name]
    if sub.empty: continue
    mu,g=gaps(sub)
    preserved = (g.get("edge-linear",1)>0.02) and (abs(g.get("ordered-edge",1))<0.05) and (g.get("edge-deep",1)>0.02)
    summary.append(dict(variant=name,axis=axis,
        **{a:round(float(mu.get(a,float("nan"))),4) for a in CFG.arms},
        **{k:round(v2,4) for k,v2 in g.items()},
        ordering_preserved=bool(preserved)))
SUM=pd.DataFrame(summary); savetable(SUM,"phase23_summary.csv")
pd.set_option("display.width",200,"display.max_columns",30)
print("Per-variant arm accuracies and the three diagnostic gaps:\n")
print(SUM.to_string(index=False))
print("\nordering_preserved = (edge-linear>0.02) AND (|ordered-edge|<0.05) AND (edge-deep>0.02)")
npres=int(SUM.ordering_preserved.sum())
print(f"\nOrdering preserved in {npres}/{len(SUM)} architecture variants.")
if npres==len(SUM):
    print("=> the central frozen-reservoir ordering is NOT an artifact of width, normalization,")
    print("   readout, or depth: nonlinearity helps, criticality holds no privileged position,")
    print("   and deep chaos collapses, under every variation tested.")

saved /content/drive/My Drive/ChaoSSM/results/phase23_robustness/phase23_summary.csv
Per-variant arm accuracies and the three diagnostic gaps:

    variant     axis  ordered   edge   deep  linear  edge-linear  ordered-edge  edge-deep  ordering_preserved
   baseline baseline   0.3550 0.3297 0.2182  0.2504       0.0793        0.0253     0.1115                True
   width=32    width   0.2966 0.2799 0.2024  0.2403       0.0396        0.0167     0.0775                True
  width=128    width   0.4869 0.4222 0.2340  0.2543       0.1679        0.0648     0.1882               False
  norm=none     norm   0.3113 0.2989 0.2180  0.2488       0.0502        0.0123     0.0809                True
norm=zscore     norm   0.3737 0.3403 0.2182  0.2527       0.0876        0.0334     0.1221                True
readout=mlp  readout   0.7488 0.7060 0.3028  0.5164       0.1896        0.0428     0.4032                True
    depth=2    depth   0.4050 0.3382 0.1797  0.2482       0.0900        0.0667     0.1

## Figure (600 dpi)

In [8]:
order=[v[0] for v in variants if (RES.variant==v[0]).any()]
fig,ax=plt.subplots(1,2,figsize=(13,4.8))
armc={"ordered":"#2E7D32","edge":"#4C72B0","deep":"#B71C1C","linear":"#9E9E9E"}
x=np.arange(len(order)); w=0.2
for i,a in enumerate(CFG.arms):
    vals=[RES[(RES.variant==o)&(RES.arm==a)].acc.mean() for o in order]
    sds=[RES[(RES.variant==o)&(RES.arm==a)].acc.std() for o in order]
    ax[0].bar(x+(i-1.5)*w,vals,w,yerr=sds,capsize=2,label=a,color=armc.get(a,"#777"))
ax[0].set_xticks(x); ax[0].set_xticklabels(order,rotation=30,ha="right",fontsize=8)
ax[0].set_ylabel("held-out accuracy"); ax[0].legend(fontsize=8,ncol=2)
ax[0].set_title("(a) Arm accuracy across architecture variants",fontsize=11.5); ax[0].grid(axis="y",alpha=.25)
# (b) the three gaps per variant
gk=["edge-linear","ordered-edge","edge-deep"]; gc=["#1b7837","#777777","#B71C1C"]
for j,k in enumerate(gk):
    vals=[]
    for o in order:
        mu=RES[RES.variant==o].groupby("arm").acc.mean()
        if k=="edge-linear": vals.append(mu.get("edge",np.nan)-mu.get("linear",np.nan))
        elif k=="ordered-edge": vals.append(mu.get("ordered",np.nan)-mu.get("edge",np.nan))
        else: vals.append(mu.get("edge",np.nan)-mu.get("deep",np.nan))
    ax[1].bar(x+(j-1)*0.26,vals,0.26,label=k,color=gc[j])
ax[1].axhline(0,color="#333",lw=0.8); ax[1].axhline(0.02,ls=":",color="#1b7837",lw=1)
ax[1].set_xticks(x); ax[1].set_xticklabels(order,rotation=30,ha="right",fontsize=8)
ax[1].set_ylabel("accuracy gap"); ax[1].legend(fontsize=8)
ax[1].set_title("(b) Diagnostic gaps (edge−linear>0, ordered−edge≈0, edge−deep>0)",fontsize=11)
fig.tight_layout(); savefig(fig,"phase23_robustness.png")

saved /content/drive/My Drive/ChaoSSM/results/phase23_robustness/phase23_robustness.png


## Rescue if Drive was not mounted

In [9]:
import shutil
if not RESULTS.startswith("/content/drive"):
    z=shutil.make_archive("/content/phase23_results","zip",RESULTS); print("archive:",z)
    try:
        from google.colab import files; files.download(z)
    except Exception as e: print("download",z,"manually:",e)
else: print("on Drive:",RESULTS)

on Drive: /content/drive/My Drive/ChaoSSM/results/phase23_robustness
